# Тестирование модели для распознования тем Уровень 1

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers scikit-learn tqdm pandas torch

In [3]:
import torch
import joblib
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./drive/MyDrive/s7_level1_model_tune_tune"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 256

In [7]:

print(f"Используется устройство: {DEVICE}")

if not os.path.isdir(MODEL_PATH):
    raise FileNotFoundError(
        f"Папка с моделью не найдена по пути: '{MODEL_PATH}'. "
        "Убедитесь, что скрипт запущен в правильной директории."
    )

print("🔹 Загрузка токенизатора, модели и энкодера...")

try:

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)

    label_encoder = joblib.load(os.path.join(MODEL_PATH, "level1_label_encoder.joblib"))

    model.eval()

    print("✅ Компоненты успешно загружены.")

except Exception as e:
    print(f"❗️ Произошла ошибка при загрузке компонентов: {e}")
    exit()


def predict_level1(text: str):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LEN
    )

    inputs = {key: val.to(DEVICE) for key, val in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class_id = torch.argmax(logits, dim=-1).item()


    predicted_label = label_encoder.inverse_transform([predicted_class_id])[0]

    return predicted_label


Используется устройство: cuda
🔹 Загрузка токенизатора, модели и энкодера...
✅ Компоненты успешно загружены.


In [9]:
test_queries = [
    "Расстроен работой программы лояльности, бонусные мили начисляются с задержкой. Накопленные бонусы невозможно использовать.",
    "Почему мой рейс задерживается уже на 2 часа?",
    "Не могу понять систему списания миль за неактивность!",
    "Где можно узнать статус моего багажа, я уже давно прилетел, а багажа нет",
    "Как купить билет?",
    "Привет, рад общению!",
    "Примут ли паспорт с размытой печатью?",
    "Почему ваш сайт так долго загружается?",
    "Сколько стоит дополнительный багаж?",
    "Можно ли провезти велосипед в багаже?",
    "Привет, а у вас есть скидки для студентов?",
    "Добрый день! У меня сгорело 5000 миль! Почему мне не пришло уведомление?",
    "Нужна ли справка об отсутствии COVID?",
    "Прошу уточнить, вылетел ли рейс S7 2525 вовремя? В системе отображаются conflicting данные, нужна точная информация.",
    "Какие правила провоза животных на борту?",
    "Какие документы для полета беременной?",
    "Добрый день",
    "Заметила что цены растут если часто ищешь один и тот же рейс. Это специальная система или просто кажется?",
    "Почему при покупке билета за мили все равно нужно платить какие-то сборы? Это скрытые платежи!",
    "Мой телефон Samsung начал самопроизвольно выключаться и включаться. Что это может быть? Сломался аккумулятор или нужно перепрошивать?",
    "Подскажите, как выбрать спелый арбуз? Стучу по ним, как советуют, но всё равно попадаются невкусные и бледные внутри.",
    "Хочу выразить благодарность команде стюардесс за внимание и заботу о пассажирах. Их профессиональное обслуживание сделало путешествие комфортным.",
    "Хочу поблагодарить службу безопасности аэропорта за оперативность в ситуации с потерей документов. Все было быстро найдено и возвращено.",
    "Питание - это просто позор для авиакомпании! Все блюда были холодными, порции микроскопическими, а качество продуктов отвратительным. Развлекательная система не работала, экраны мерцали. Бортпроводники совершенно не следили за порядком в салоне.",
    "Очень грязно на борту"
]

for query in test_queries:
    prediction = predict_level1(query)
    print(f"Вопрос: '{query}'")
    print(f"Предсказанная категория Level 1: '{prediction}'")
    print("-" * 50)


Вопрос: 'Расстроен работой программы лояльности, бонусные мили начисляются с задержкой. Накопленные бонусы невозможно использовать.'
Предсказанная категория Level 1: 'Обратная связь'
--------------------------------------------------
Вопрос: 'Почему мой рейс задерживается уже на 2 часа?'
Предсказанная категория Level 1: 'Вылет Прилет'
--------------------------------------------------
Вопрос: 'Не могу понять систему списания миль за неактивность!'
Предсказанная категория Level 1: 'Лояльность и продажи'
--------------------------------------------------
Вопрос: 'Где можно узнать статус моего багажа, я уже давно прилетел, а багажа нет'
Предсказанная категория Level 1: 'Вылет Прилет'
--------------------------------------------------
Вопрос: 'Как купить билет?'
Предсказанная категория Level 1: 'Сервисные'
--------------------------------------------------
Вопрос: 'Привет, рад общению!'
Предсказанная категория Level 1: 'Сервисные'
--------------------------------------------------
Вопрос: 